# VSO, Kubernetes Secrets Engine y GitHub Actions OIDC

Este notebook construye un flujo completo sobre el clúster EKS:

1. Instala Vault Secrets Operator (VSO).
2. Configura Kubernetes Authentication en el **root namespace** de Vault.
3. Despliega `VaultConnection`, `VaultAuth`, `VaultStaticSecret` y una carga de prueba en `default`.
4. Configura Kubernetes Secrets Engine en root para emitir credenciales temporales a GitHub Actions.
5. Crea un auth method OIDC y un rol específicos para la pipeline.
6. La pipeline crea un namespace delegado de Vault, configura en él Kubernetes Authentication y despliega una aplicación NGINX en un namespace nuevo de Kubernetes.

El contenido creado en root actúa como **bootstrap**. No se guardan tokens de Kubernetes, reviewer JWT ni secretos de aplicación en Git.

## 1. Variables y credenciales

Carga Vault desde `.env`, renueva las credenciales de laboratorio mediante Doormat y selecciona el clúster EKS. Las credenciales AWS solo se importan en el kernel.

In [ ]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if ENV_FILE is None:
    raise FileNotFoundError("No se encontró el fichero .env")
load_dotenv(ENV_FILE, override=True)

required = ("VAULT_ADDR", "VAULT_TOKEN", "VAULT_CACERT")
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Variables requeridas ausentes: {', '.join(missing)}")

subprocess.run(["doormat", "login", "-f"], check=True)
aws_env = subprocess.run(
    ["bash", "-lc", 'eval "$(doormat aws -a aws_jose.merchan_test export)" && env -0'],
    check=True,
    capture_output=True,
).stdout
for entry in aws_env.split(b"\0"):
    if entry.startswith(b"AWS_") and b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

settings = {
    "AWS_REGION": "eu-central-1",
    "EKS_CLUSTER_NAME": "eks-infra-dev",
    "WORKDIR": "/tmp/vault-k8s-engine",
    "VSO_NAMESPACE": "vault-secrets-operator",
    "VSO_VERSION": "1.5.0",
    "VAULT_K8S_AUTH_PATH": "kubernetes",
    "VAULT_K8S_ENGINE_PATH": "kubernetes",
    "GHA_OIDC_AUTH_PATH": "github-k8s-demo",
    "GHA_K8S_POLICY": "gha-k8s-bootstrap",
    "GHA_K8S_ROLE": "gha-k8s-bootstrap-main",
    "GHA_K8S_ENGINE_ROLE": "github-actions-deployer",
    "REPO_FULL_NAME": "jm-merchan/Vault_Use_Cases_Example_202607",
    "VAULT_CHILD_NAMESPACE": "gha-demo",
    "APP_NAMESPACE": "gha-vso-demo",
}
os.environ.update(settings)
Path(settings["WORKDIR"]).mkdir(parents=True, exist_ok=True)

subprocess.run([
    "aws", "eks", "update-kubeconfig",
    "--region", settings["AWS_REGION"],
    "--name", settings["EKS_CLUSTER_NAME"],
], check=True)
print(f"EKS: {settings['EKS_CLUSTER_NAME']} ({settings['AWS_REGION']})")
print(f"Vault: {os.environ['VAULT_ADDR']}")
print(f"Workflow: .github/workflows/vault-k8s-engine-vso.yml")

## 2. Prerrequisitos

Comprueba conectividad con EKS y Vault antes de realizar cambios.

In [ ]:
%%bash
set -euo pipefail

kubectl cluster-info
kubectl get nodes
vault status
kubectl -n vault get statefulset,pods,service
command -v jq >/dev/null
command -v helm >/dev/null
command -v gh >/dev/null
echo 'OK: prerrequisitos disponibles.'

## 3. Instalación de Vault Secrets Operator

Instala VSO mediante el chart oficial de HashiCorp y espera a que el controlador esté disponible. La instalación es idempotente.

In [ ]:
%%bash
set -euo pipefail

helm repo add hashicorp https://helm.releases.hashicorp.com --force-update
helm repo update
helm upgrade --install vault-secrets-operator hashicorp/vault-secrets-operator \
  --namespace "${VSO_NAMESPACE}" \
  --create-namespace \
  --version "${VSO_VERSION}" \
  --wait \
  --timeout 5m

kubectl -n "${VSO_NAMESPACE}" rollout status \
  deployment/vault-secrets-operator-controller-manager --timeout=5m
kubectl api-resources --api-group=secrets.hashicorp.com


## 4. Bootstrap root: Kubernetes Authentication

El service account de Vault recibe `system:auth-delegator` para realizar TokenReview. El auth method usa el API interno de Kubernetes y el JWT local de los pods de Vault, por lo que no se persiste ningún reviewer JWT manual.

El rol `vso-bootstrap` acepta únicamente el service account `default` del namespace Kubernetes `default`.

In [ ]:
%%bash
set -euo pipefail

kubectl create clusterrolebinding vault-token-reviewer \
  --clusterrole=system:auth-delegator \
  --serviceaccount=vault:vault \
  --dry-run=client -o yaml | kubectl apply -f -

if ! vault auth list -format=json | jq -e 'has("kubernetes/")' >/dev/null; then
  vault auth enable -path="${VAULT_K8S_AUTH_PATH}" kubernetes
fi
vault write "auth/${VAULT_K8S_AUTH_PATH}/config" \
  kubernetes_host="https://kubernetes.default.svc:443"

cat > "${WORKDIR}/vso-bootstrap.hcl" <<'EOF'
path "secret/data/vso-bootstrap/*" {
  capabilities = ["read"]
}
path "secret/metadata/vso-bootstrap/*" {
  capabilities = ["read", "list"]
}
EOF
vault policy write vso-bootstrap "${WORKDIR}/vso-bootstrap.hcl"
vault write "auth/${VAULT_K8S_AUTH_PATH}/role/vso-bootstrap" \
  bound_service_account_names=default \
  bound_service_account_namespaces=default \
  audience=vault \
  token_policies=vso-bootstrap \
  token_ttl=1h

if ! vault secrets list -format=json | jq -e 'has("secret/")' >/dev/null; then
  vault secrets enable -path=secret kv-v2
fi
vault kv put secret/vso-bootstrap/application \
  message='VSO authenticated with Kubernetes in Vault root' \
  environment='EKS bootstrap'

## 5. Bootstrap root: VaultConnection, VaultAuth y aplicación de prueba

VSO sincroniza el secreto de root a `bootstrap-application-secret`. La aplicación consume el secreto mediante variables de entorno y se reinicia cuando VSO detecta una nueva versión.

In [ ]:
%%bash
set -euo pipefail

cat > "${WORKDIR}/root-vso-bootstrap.yaml" <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultConnection
metadata:
  name: vault-root
  namespace: default
spec:
  address: ${VAULT_ADDR}
  skipTLSVerify: true
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultAuth
metadata:
  name: vault-root
  namespace: default
spec:
  vaultConnectionRef: vault-root
  method: kubernetes
  mount: ${VAULT_K8S_AUTH_PATH}
  kubernetes:
    role: vso-bootstrap
    serviceAccount: default
    audiences:
      - vault
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  name: bootstrap-application
  namespace: default
spec:
  vaultAuthRef: vault-root
  type: kv-v2
  mount: secret
  path: vso-bootstrap/application
  refreshAfter: 30s
  destination:
    create: true
    overwrite: true
    name: bootstrap-application-secret
    transformation:
      excludeRaw: true
  rolloutRestartTargets:
    - kind: Deployment
      name: bootstrap-secret-consumer
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: bootstrap-secret-consumer
  namespace: default
spec:
  replicas: 1
  selector:
    matchLabels:
      app: bootstrap-secret-consumer
  template:
    metadata:
      labels:
        app: bootstrap-secret-consumer
    spec:
      containers:
        - name: consumer
          image: busybox:1.37
          command: ["sh", "-c", "test -n \"\${SECRET_MESSAGE}\"; echo \"Secret injected: \${SECRET_MESSAGE}\"; sleep 3600"]
          env:
            - name: SECRET_MESSAGE
              valueFrom:
                secretKeyRef:
                  name: bootstrap-application-secret
                  key: message
EOF

kubectl apply -f "${WORKDIR}/root-vso-bootstrap.yaml"

In [ ]:
%%bash
set -euo pipefail

for attempt in $(seq 1 60); do
  ready=$(kubectl -n default get vaultstaticsecret bootstrap-application \
    -o jsonpath='{.status.conditions[?(@.type=="Ready")].status}' 2>/dev/null || true)
  [[ "${ready}" == "True" ]] && break
  [[ "${attempt}" == "60" ]] && {
    kubectl -n default describe vaultstaticsecret bootstrap-application
    exit 1
  }
  sleep 5
done
kubectl -n default rollout status deployment/bootstrap-secret-consumer --timeout=5m
kubectl -n default logs deployment/bootstrap-secret-consumer
kubectl -n default get vaultconnection,vaultauth,vaultstaticsecret,secret,pod
echo 'OK: VSO root autenticado mediante Kubernetes y secreto consumido como env.'

## 6. Bootstrap root: Kubernetes Secrets Engine

Se crea un service account técnico para el secrets engine y otro para la pipeline. Vault conserva el token técnico dentro de su storage; el fichero temporal local se elimina inmediatamente.

La credencial dinámica de GitHub Actions tiene permisos limitados para crear el namespace de la demo y gestionar Deployments, Services, Secrets, ConfigMaps, service accounts y CRDs de VSO. No recibe `cluster-admin`.

In [ ]:
%%bash
set -euo pipefail
umask 077

cat > "${WORKDIR}/kubernetes-engine-rbac.yaml" <<'EOF'
apiVersion: v1
kind: ServiceAccount
metadata:
  name: vault-k8s-engine
  namespace: vault
---
apiVersion: v1
kind: Secret
metadata:
  name: vault-k8s-engine-token
  namespace: vault
  annotations:
    kubernetes.io/service-account.name: vault-k8s-engine
type: kubernetes.io/service-account-token
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: gha-deployer
  namespace: vault
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: vault-k8s-engine-token-issuer
  namespace: vault
rules:
  - apiGroups: [""]
    resources: ["serviceaccounts"]
    verbs: ["get"]
  - apiGroups: [""]
    resources: ["serviceaccounts/token"]
    verbs: ["create"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: vault-k8s-engine-token-issuer
  namespace: vault
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: vault-k8s-engine-token-issuer
subjects:
  - kind: ServiceAccount
    name: vault-k8s-engine
    namespace: vault
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: gha-vso-deployer
rules:
  - apiGroups: [""]
    resources: ["namespaces"]
    verbs: ["create", "get", "list", "watch"]
  - apiGroups: [""]
    resources: ["configmaps", "secrets", "serviceaccounts", "services", "pods", "pods/log", "events"]
    verbs: ["create", "get", "list", "watch", "update", "patch", "delete"]
  - apiGroups: ["apps"]
    resources: ["deployments", "deployments/status", "replicasets"]
    verbs: ["create", "get", "list", "watch", "update", "patch", "delete"]
  - apiGroups: ["secrets.hashicorp.com"]
    resources: ["vaultconnections", "vaultauths", "vaultstaticsecrets"]
    verbs: ["create", "get", "list", "watch", "update", "patch", "delete"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: gha-vso-deployer
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: gha-vso-deployer
subjects:
  - kind: ServiceAccount
    name: gha-deployer
    namespace: vault
EOF
kubectl apply -f "${WORKDIR}/kubernetes-engine-rbac.yaml"

for attempt in $(seq 1 30); do
  engine_jwt=$(kubectl -n vault get secret vault-k8s-engine-token \
    -o jsonpath='{.data.token}' 2>/dev/null | base64 --decode || true)
  [[ -n "${engine_jwt}" ]] && break
  sleep 2
done
test -n "${engine_jwt}"
kubectl -n kube-system get configmap kube-root-ca.crt \
  -o jsonpath='{.data.ca\.crt}' > "${WORKDIR}/eks-ca.crt"

if ! vault secrets list -format=json | jq -e 'has("kubernetes/")' >/dev/null; then
  vault secrets enable -path="${VAULT_K8S_ENGINE_PATH}" kubernetes
fi
vault write "${VAULT_K8S_ENGINE_PATH}/config" \
  kubernetes_host="https://kubernetes.default.svc:443" \
  kubernetes_ca_cert=@"${WORKDIR}/eks-ca.crt" \
  service_account_jwt="${engine_jwt}"
unset engine_jwt
rm -f "${WORKDIR}/eks-ca.crt"

vault write "${VAULT_K8S_ENGINE_PATH}/roles/${GHA_K8S_ENGINE_ROLE}" \
  allowed_kubernetes_namespaces=vault \
  service_account_name=gha-deployer \
  token_default_ttl=1h \
  token_max_ttl=2h
vault read "${VAULT_K8S_ENGINE_PATH}/roles/${GHA_K8S_ENGINE_ROLE}"

## 7. Rol OIDC de GitHub Actions

Este notebook crea su propio auth method OIDC `github-k8s-demo/`. Después añade un rol específico para esta pipeline y una política de bootstrap:

- Puede solicitar únicamente el rol dinámico `github-actions-deployer` del Kubernetes Secrets Engine.
- Puede crear y configurar únicamente el namespace Vault `gha-demo`.
- No recibe acceso general a otros namespaces ni engines de Vault.

In [ ]:
%%bash
set -euo pipefail

if ! vault auth list -format=json | jq -e --arg path "${GHA_OIDC_AUTH_PATH}/" 'has($path)' >/dev/null; then
  vault auth enable -path="${GHA_OIDC_AUTH_PATH}" jwt
fi
vault write "auth/${GHA_OIDC_AUTH_PATH}/config" \
  oidc_discovery_url="https://token.actions.githubusercontent.com" \
  bound_issuer="https://token.actions.githubusercontent.com"

cat > "${WORKDIR}/${GHA_K8S_POLICY}.hcl" <<EOF
path "${VAULT_K8S_ENGINE_PATH}/creds/${GHA_K8S_ENGINE_ROLE}" {
  capabilities = ["update"]
}
path "sys/leases/revoke" {
  capabilities = ["update"]
}
path "sys/namespaces/${VAULT_CHILD_NAMESPACE}" {
  capabilities = ["create", "update", "read"]
}
path "${VAULT_CHILD_NAMESPACE}/sys/mounts" {
  capabilities = ["read"]
}
path "${VAULT_CHILD_NAMESPACE}/sys/mounts/secret" {
  capabilities = ["create", "update", "read", "sudo"]
}
path "${VAULT_CHILD_NAMESPACE}/sys/auth/kubernetes" {
  capabilities = ["create", "update", "read", "sudo"]
}
path "${VAULT_CHILD_NAMESPACE}/sys/auth" {
  capabilities = ["read"]
}
path "${VAULT_CHILD_NAMESPACE}/auth/kubernetes/config" {
  capabilities = ["create", "update", "read"]
}
path "${VAULT_CHILD_NAMESPACE}/auth/kubernetes/role/vso-application" {
  capabilities = ["create", "update", "read"]
}
path "${VAULT_CHILD_NAMESPACE}/sys/policies/acl/vso-application" {
  capabilities = ["create", "update", "read"]
}
path "${VAULT_CHILD_NAMESPACE}/secret/data/application/ui" {
  capabilities = ["create", "update", "read"]
}
EOF
vault policy write "${GHA_K8S_POLICY}" "${WORKDIR}/${GHA_K8S_POLICY}.hcl"

cat > "${WORKDIR}/role-${GHA_K8S_ROLE}.json" <<EOF
{
  "role_type": "jwt",
  "user_claim": "actor",
  "bound_audiences": "https://github.com/jm-merchan",
  "bound_claims_type": "glob",
  "bound_claims": {
    "repository": "${REPO_FULL_NAME}",
    "ref": "refs/heads/main"
  },
  "token_policies": "${GHA_K8S_POLICY}",
  "token_ttl": "30m"
}
EOF
vault write "auth/${GHA_OIDC_AUTH_PATH}/role/${GHA_K8S_ROLE}" @"${WORKDIR}/role-${GHA_K8S_ROLE}.json"
vault read "auth/${GHA_OIDC_AUTH_PATH}/role/${GHA_K8S_ROLE}"

## 8. Variables del repositorio y pipeline

Publica únicamente datos no secretos como variables de GitHub: URL de Vault, auth path, nombre del rol, endpoint EKS y CA pública del clúster. El workflow está versionado en `.github/workflows/vault-k8s-engine-vso.yml`.

In [ ]:
%%bash
set -euo pipefail

EKS_SERVER=$(kubectl config view --raw --minify -o jsonpath='{.clusters[0].cluster.server}')
EKS_CA_DATA=$(kubectl config view --raw --minify -o jsonpath='{.clusters[0].cluster.certificate-authority-data}')

gh variable set VAULT_ADDR --repo "${REPO_FULL_NAME}" --body "https://vault.jose-merchan.sbx.hashidemos.io"
gh variable set VAULT_AUTH_PATH --repo "${REPO_FULL_NAME}" --body "${GHA_OIDC_AUTH_PATH}"
gh variable set VAULT_K8S_AUTH_ROLE --repo "${REPO_FULL_NAME}" --body "${GHA_K8S_ROLE}"
gh variable set EKS_SERVER --repo "${REPO_FULL_NAME}" --body "${EKS_SERVER}"
gh variable set EKS_CA_DATA --repo "${REPO_FULL_NAME}" --body "${EKS_CA_DATA}"

gh variable list --repo "${REPO_FULL_NAME}" | \
  grep -E '^(VAULT_ADDR|VAULT_AUTH_PATH|VAULT_K8S_AUTH_ROLE|EKS_SERVER|EKS_CA_DATA)[[:space:]]'
test -f .github/workflows/vault-k8s-engine-vso.yml
echo 'OK: variables no sensibles configuradas y workflow presente.'

## 9. Qué hace la pipeline

El workflow realiza esta secuencia:

1. Solicita un ID token a GitHub OIDC y autentica en el auth method exclusivo `auth/github-k8s-demo`.
2. Solicita a `kubernetes/creds/github-actions-deployer` un token Kubernetes de una hora.
3. Crea el namespace Vault `gha-demo`, su KV v2, política y Kubernetes auth method.
4. Crea el namespace Kubernetes `gha-vso-demo`.
5. Despliega `VaultConnection`, `VaultAuth` y `VaultStaticSecret`.
6. Despliega NGINX usando el Secret como variables de entorno en un init container que renderiza la interfaz.
7. Configura `rolloutRestartTargets` para que VSO reinicie el Deployment al cambiar el secreto.
8. Expone la interfaz mediante un LoadBalancer público y revoca la credencial dinámica al terminar.

In [ ]:
%%bash
set -euo pipefail

# El workflow debe estar en GitHub antes de ejecutar esta celda.
gh workflow run vault-k8s-engine-vso.yml \
  --repo "${REPO_FULL_NAME}" \
  --ref main \
  -f secret_message='Rotated securely by GitHub OIDC and Vault'
sleep 5
gh run list --repo "${REPO_FULL_NAME}" \
  --workflow vault-k8s-engine-vso.yml \
  --limit 3

## 10. Validación posterior

Después de completar el workflow, verifica el estado VSO, el rollout y la dirección pública.

In [ ]:
%%bash
set -euo pipefail

kubectl -n "${APP_NAMESPACE}" get vaultconnection,vaultauth,vaultstaticsecret
kubectl -n "${APP_NAMESPACE}" get deployment,pods,service
kubectl -n "${APP_NAMESPACE}" rollout status deployment/secret-ui --timeout=5m
endpoint=$(kubectl -n "${APP_NAMESPACE}" get service secret-ui \
  -o jsonpath='{.status.loadBalancer.ingress[0].hostname}')
test -n "${endpoint}"
echo "URL pública: http://${endpoint}"
curl --silent --show-error --fail --retry 12 --retry-delay 10 \
  "http://${endpoint}" | grep -q 'Secret delivered.'
echo 'OK: interfaz NGINX disponible y secreto renderizado.'

## 11. Prueba de reload

Actualiza el secreto en el namespace Vault delegado y verifica que VSO cambia el Secret y reinicia NGINX. La pipeline ya configura `refreshAfter: 30s` y `rolloutRestartTargets`.

In [ ]:
%%bash
set -euo pipefail

before=$(kubectl -n "${APP_NAMESPACE}" get deployment secret-ui \
  -o jsonpath='{.spec.template.metadata.annotations.vso\.secrets\.hashicorp\.com/restartedAt}' 2>/dev/null || true)
VAULT_NAMESPACE="${VAULT_CHILD_NAMESPACE}" vault kv put secret/application/ui \
  message="Reload validated at $(date -u +%FT%TZ)" \
  environment='Notebook 12 reload test'

for attempt in $(seq 1 24); do
  after=$(kubectl -n "${APP_NAMESPACE}" get deployment secret-ui \
    -o jsonpath='{.spec.template.metadata.annotations.vso\.secrets\.hashicorp\.com/restartedAt}' 2>/dev/null || true)
  [[ -n "${after}" && "${after}" != "${before}" ]] && break
  [[ "${attempt}" == "24" ]] && exit 1
  sleep 5
done
kubectl -n "${APP_NAMESPACE}" rollout status deployment/secret-ui --timeout=5m
echo "OK: reload detectado (${after})."

## 12. Cleanup opcional

Está bloqueado por defecto. Elimina la aplicación, los recursos bootstrap, VSO y toda la configuración Vault creada exclusivamente por este notebook, incluido su auth method OIDC.

In [ ]:
%%bash
set -euo pipefail

CONFIRM_CLEANUP=DELETE_NOTEBOOK_12_DEMO
if [[ "${CONFIRM_CLEANUP}" != "DELETE_NOTEBOOK_12_DEMO" ]]; then
  echo 'Cleanup omitido. Usa CONFIRM_CLEANUP=DELETE_NOTEBOOK_12_DEMO para habilitarlo.'
  exit 0
fi

kubectl delete namespace "${APP_NAMESPACE}" --ignore-not-found --wait
kubectl -n default delete deployment bootstrap-secret-consumer --ignore-not-found
kubectl -n default delete vaultstaticsecret bootstrap-application --ignore-not-found
kubectl -n default delete vaultauth vault-root --ignore-not-found
kubectl -n default delete vaultconnection vault-root --ignore-not-found
kubectl delete clusterrolebinding gha-vso-deployer vault-token-reviewer --ignore-not-found
kubectl delete clusterrole gha-vso-deployer --ignore-not-found
kubectl -n vault delete rolebinding vault-k8s-engine-token-issuer --ignore-not-found
kubectl -n vault delete role vault-k8s-engine-token-issuer --ignore-not-found
kubectl -n vault delete secret vault-k8s-engine-token --ignore-not-found
kubectl -n vault delete serviceaccount vault-k8s-engine gha-deployer --ignore-not-found
helm uninstall vault-secrets-operator -n "${VSO_NAMESPACE}" --ignore-not-found
kubectl delete namespace "${VSO_NAMESPACE}" --ignore-not-found

vault namespace delete "${VAULT_CHILD_NAMESPACE}" 2>/dev/null || true
vault policy delete "${GHA_K8S_POLICY}" 2>/dev/null || true
vault policy delete vso-bootstrap 2>/dev/null || true
vault auth disable "${GHA_OIDC_AUTH_PATH}" 2>/dev/null || true
vault secrets disable "${VAULT_K8S_ENGINE_PATH}" 2>/dev/null || true
vault auth disable "${VAULT_K8S_AUTH_PATH}" 2>/dev/null || true
vault kv metadata delete secret/vso-bootstrap/application 2>/dev/null || true
echo 'Cleanup completado.'